In [29]:
import re
import pandas as pd
import requests
import yaml
import time
import random
import uuid
import os

In [30]:
# Load credentials from creds.yaml
with open("../creds.yaml", "r") as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as e:
        print(f"Error loading YAML file: {e}")

# initialize SocialCrawl API key to use for requests
socialcrawl_key = creds["socialcrawl"]["api_key"]

In [31]:
# Make sure Facebook API request was successful
# Got the following code from SocialCrawl support because calling the Facebook API directly wasn't working
# Code credit: 
# https://www.socialcrawl.dev/dashboard/api/explorer?call=%2Fv1%2Ffacebook%2Fprofile&url=https%3A%2F%2Fwww.facebook.com%2FMeta

TIMEOUT_S = 14
MAX_RETRIES = 3
URL = "https://www.socialcrawl.dev/v1/facebook/profile"

def backoff(attempt):
    # Exponential backoff with jitter: about 1 s, 2 s, then 4 s.
    return 2 ** attempt * (0.5 + random.random() / 2)

def socialcrawl(params):
    # One Idempotency-Key per request, reused on every retry, so a retried call is never charged twice.
    idempotency_key = str(uuid.uuid4())
    for attempt in range(MAX_RETRIES + 1):
        try:
            res = requests.get(
                URL,
                params=params,
                headers={"x-api-key": socialcrawl_key, "Idempotency-Key": idempotency_key},
                timeout=TIMEOUT_S,
            )
            body = res.json()
        except (requests.RequestException, ValueError):
            # A timeout, a dropped connection or a non-JSON gateway page: retry with the same key.
            if attempt == MAX_RETRIES:
                raise
            time.sleep(backoff(attempt))
            continue
        if body["success"]:
            return body
        error = body["error"]
        # Branch on error["type"], never on the message text.
        if error["type"] == "RESOURCE_NOT_FOUND":
            return None  # refunded: nothing to read
        retry_after = res.headers.get("Retry-After")
        # SERVICE_UNAVAILABLE without Retry-After is a withdrawn endpoint: permanent.
        permanent = error["type"] == "SERVICE_UNAVAILABLE" and retry_after is None
        if not error.get("retryable") or permanent or attempt == MAX_RETRIES:
            raise RuntimeError(f"{error['type']}: {error['message']} (request_id {body['request_id']})")
        # Retry-After (seconds) wins; otherwise back off.
        time.sleep(float(retry_after) if retry_after else backoff(attempt))


params = {
    "url": "https://www.facebook.com/Meta",
}
body = socialcrawl(params)
if body is not None:
    print(body["data"])
    print("Facebook API Status: ", body["success"])


{'author': {'id': '100080376596424', 'username': 'Meta', 'display_name': 'Meta', 'avatar_url': 'https://scontent-sea5-2.xx.fbcdn.net/v/t39.30808-1/825292571_1146841521338390_8821495995170295283_n.jpg?stp=c0.11.852.852a_cp6_dst-jpg_tt6&cstp=mx852x852&ctp=s480x480&_nc_cat=1&ccb=1-7&_nc_sid=2d3e12&_nc_ohc=4ONzBi8i_QgQ7kNvwG0TFhl&_nc_oc=Adojbwv-mhVmCnZBh-BGsU2Cq7RSgfGuCvpGLSILsn8LH1ZNubukIDmgkU1wGnHQ8M4&_nc_zt=24&_nc_ht=scontent-sea5-2.xx&_nc_gid=XIYNTt4FABWSW5MEoqrNtg&_nc_ss=78180&oh=00_AQQsFcEGHUilm4s7iBIR9PuLarnD0q_6U3-pwAa09Se4aQ&oe=6B0986C1', 'bio': 'Connect with what you love to make things happen.', 'verified': None, 'followers': 106817204, 'following': None, 'posts_count': None, 'likes_count': 106817204, 'url': 'https://www.facebook.com/Meta', 'private': None, 'joined_at': '2020-01-31', 'ext': {'public_email': None, 'public_phone': None, 'business_category': 'Company', 'website': 'https://www.meta.com/connect/', 'cover_url': 'https://scontent-sea5-1.xx.fbcdn.net/v/t39.30808-6/82529

In [32]:
# Downloading recent posts from Instagram (SocialCrawl gives 100 credits TOTAL)
def get_posts_facebook(url, max_pages=3):
    """ 
    Helper function to get posts from an Instagram account using the SocialCrawl API.
    Args:
        handle (str): Instagram username to scrape posts from.
        max_pages (int): Maximum number of pages to scrape. Each page contains up to 12 posts. Default is 3 pages (36 posts).
    Returns:
        list: A list of posts retrieved from the Instagram account.
    """
    rows, cursor = [], None
    for _ in range(max_pages):
        params = {"url": url}
        if cursor:
            params["cursor"] = cursor
        res = requests.get(
            "https://www.socialcrawl.dev/v1/facebook/profile/posts",
            params=params, headers={"x-api-key": socialcrawl_key}, timeout=30,
        )
        res.raise_for_status()
        body = res.json()
        data = body["data"]
        if len(rows) == 0:
            print(body.get("credits_remaining"))
            print(str(data)[:1500])
        items = data if isinstance(data, list) else (data.get("items") or [])
        rows.extend(items)
        pagination = body.get("pagination") or {}
        cursor = pagination.get("next_cursor")
        if not cursor:
            break
        time.sleep(random.uniform(1, 3))
    return rows


In [33]:
# Import club and university information from the spreadsheet we created
social_media = pd.read_csv("../Data/Social Media for University - data.csv")
social_media.columns = social_media.columns.str.strip()

for column in ["university", "club", "instagram_handle", "facebook_handle", "instagram_link", "facebook_link"]:
    social_media[column] = social_media[column].fillna("").astype(str).str.strip()

In [ ]:
facebook_rows = social_media[
    social_media["facebook_handle"].fillna("").ne("")
    & social_media["facebook_link"].fillna("").str.contains(
        "facebook.com/", case=False, regex=False
    )
]
# filtering out /group links because I couldn't scrape them with the API 
facebook_pages = facebook_rows[
    ~facebook_rows["facebook_link"]
    .fillna("")
    .str.lower()
    .str.contains("/groups/", regex=False)
].copy()

In [35]:
club = dict(zip(
    facebook_pages["facebook_link"],
    facebook_pages["club"],
))

university = dict(zip(
    facebook_pages["facebook_link"],
    facebook_pages["university"],
))

In [ ]:
#making sure the output file exists and loading previously scraped data if it does
OUT = "../Data/facebook_scraped_posts.csv"

if os.path.exists(OUT):
    already_scraped = pd.read_csv(OUT)

    # Reuse existing rows and avoid re-requesting completed handles
    save_rows = already_scraped.to_dict("records")

    if "facebook_handle" in already_scraped.columns:
        completed_urls = set(already_scraped["facebook_handle"].dropna())
    else:
        completed_urls = set()
else:
    save_rows = []
    completed_urls = set()

raw_by_url_facebook = {}
facebook_urls = facebook_pages["facebook_link"].drop_duplicates().tolist()

print("Facebook pages loaded:", facebook_urls)

# scraping posts from each Facebook page and saving them to a CSV file
for url in facebook_urls:
    if url in completed_urls:
        print("Already saved:", url)
        continue

    try:
        raw = get_posts_facebook(url, max_pages=3)
        raw_by_url_facebook[url] = raw

        match = facebook_pages.loc[
            facebook_pages["facebook_link"] == url
        ]

        university = match["university"].iloc[0] if not match.empty else ""
        club = match["club"].iloc[0] if not match.empty else ""

        for item in raw:
            post = item.get("post", item)
            content = post.get("content") or {}

            save_rows.append({
                "facebook_handle": url,
                "url": post.get("url"),
                "caption": content.get("text") or "",
                "university": university,
                "club": club,
            })

        pd.DataFrame(save_rows).drop_duplicates(
            subset=["url"]
        ).to_csv(OUT, index=False)

        completed_urls.add(url)
        # keeping track of the number of posts saved for each URL
        print("Saved", url, "-", len(raw), "posts")
    
    except Exception as error:
        # doing this to keep track of socialcrawl credits and to know which URL caused the error
        print("Stopped at:", url)
        print(type(error).__name__, error)
        break

Facebook pages loaded: ['https://www.facebook.com/cuetcomputerclub', 'https://www.facebook.com/ieeecuetsb/', 'https://www.facebook.com/ieeecscuetsb/', 'https://www.facebook.com/asrro2012/', 'https://www.facebook.com/Dhaka.University.IT.Society.DUITS/', 'https://www.facebook.com/DUCSUTeam/', 'https://www.facebook.com/bangladeshncc/', 'https://www.facebook.com/eecdu/', 'https://www.facebook.com/wrro.cuet', 'https://www.facebook.com/asce.sc.cuet', 'https://www.facebook.com/changetogether.daffodil', 'https://www.facebook.com/DIUPS.bd', 'https://www.facebook.com/diukc.official', 'https://www.facebook.com/ccdiu1', 'https://www.facebook.com/DIUBEC.BD', 'https://www.facebook.com/SBSForum', 'https://www.facebook.com/DIUBDC', 'https://www.facebook.com/DIUVSC', 'https://www.facebook.com/daffodil.bondhushava', 'https://www.facebook.com/diu.sybl', 'https://www.facebook.com/diumarketingclub', 'https://www.facebook.com/DiuCreativePark', 'https://www.facebook.com/allstarsdaffodil/', 'https://www.faceb

In [37]:
# Dictionaries: words/phrases that signal each concept (classifier)
ai_terms = [
    "ai", "a.i.", "artificial intelligence", "machine learning",
    "generative ai", "genai", "chatgpt", "gemini", "claude", "llm",
    "deep learning", "neural network", "data science", "prompt engineering",
]
event_terms = [
    "event", "seminar", "workshop", "conference", "symposium", "webinar",
    "talk", "meetup", "bootcamp", "hackathon", "summit", "masterclass",
    "competition", "register", "join us",
]

In [38]:
def make_pattern(terms):
    """Match whole words only, so 'ai' does NOT match 'said', 'maintain', 'email'.
        Args: terms (list): A list of terms to match.
        Returns: re.Pattern: A compiled regular expression pattern.
    """
    return re.compile(r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b", re.IGNORECASE)

ai_pattern = make_pattern(ai_terms)
event_pattern = make_pattern(event_terms)

In [ ]:
def classify_post(caption, instagram_handle="", facebook_handle=""):
    """ Classify a post to see if it's related to AI or an event using the dictionaries of terms (ai_terms and event_terms) 
        Args: caption (str): The caption text of the Instagram post to classify.
        instagram_handle (str): The Instagram handle of the user who posted the content.
        facebook_handle (str): The Facebook handle of the user who posted the content.
        Returns: dict: A dictionary containing classification results, including whether the post is related to AI, an event, both, 
        and the specific matches found.
    """
    text = re.sub(r"\s+", " ", str(caption or ""))
    ai_hits = sorted({m.lower() for m in ai_pattern.findall(text)})
    event_hits = sorted({m.lower() for m in event_pattern.findall(text)})
    return {
        "is_ai": len(ai_hits) > 0,
        "is_event": len(event_hits) > 0,
        "ai_event": len(ai_hits) > 0 and len(event_hits) > 0,
        "ai_matches": ai_hits,
        "event_matches": event_hits,
        "instagram_handle": instagram_handle,
        "facebook_handle": facebook_handle,
    }

In [40]:
def find_caption(item):
    """ Helper function to extract the caption text from a post item. 
        Args: item (dict): The post item from which to extract the caption.
        Returns: str: The caption text, or an empty string if not found.
    """
    post = item.get("post", item)
    content = post.get("content") or {}
    return content.get("text") or ""


posts = pd.read_csv("../Data/facebook_scraped_posts.csv")

posts = posts.rename(columns={
    "club": "club_name",
})

classified = posts.apply(
    lambda row: classify_post(
        row["caption"],
        instagram_handle="",
        facebook_handle=row["facebook_handle"],
    ),
    axis=1,
).apply(pd.Series)

posts = pd.concat([posts, classified], axis=1)

# Keep one consistent club column
posts["club"] = posts["club_name"]


In [42]:
OUT = "../Data/bangladesh_universities_ai_promotion_classified.csv"

if os.path.exists(OUT):
    existing = pd.read_csv(OUT)

    existing_urls = set(existing["url"].dropna())
    # doing this to avoid duplicates if the script is run multiple times
    posts = posts[~posts["url"].isin(existing_urls)]

posts.to_csv(
    OUT,
    mode="a",
    header=not os.path.exists(OUT),
    index=False,
)

print(len(posts), "new posts added")
posts.head(10)

324 new posts added


,facebook_handle,url,caption,university,club_name,is_ai,is_event,ai_event,ai_matches,event_matches,instagram_handle,facebook_handle,club
0,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,The event is just around the corner!\n\nPartic...,Chittagong University,CUET Computer Club,False,True,False,[],"[event, join us, register]",,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
1,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/events/10841578645828...,NaN,Chittagong University,CUET Computer Club,False,False,False,[],[],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
2,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,Got an idea that's been sitting in the back of...,Chittagong University,CUET Computer Club,True,True,True,[ai],[register],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
3,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,What if you could do more than just publish an...,Chittagong University,CUET Computer Club,False,False,False,[],[],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
4,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,HEADS UP!\n\nThe first step towards ICPC Asia ...,Chittagong University,CUET Computer Club,False,False,False,[],[],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
5,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,"কোনো তদন্ত নেই, কোনো অভিযোগ নেই, কোনো কারণ দর্...",Chittagong University,CUET Computer Club,False,False,False,[],[],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
6,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,Meet those entrusted with carrying forward a l...,Chittagong University,CUET Computer Club,False,False,False,[],[],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
7,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,IT'S LIVE!\n\nCUET Computer Club is officially...,Chittagong University,CUET Computer Club,False,True,False,[],[register],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
8,https://www.facebook.com/cuetcomputerclub,https://www.facebook.com/cuetcomputerclub/post...,"And with that, we've come to the end of the fe...",Chittagong University,CUET Computer Club,False,True,False,[],[event],,https://www.facebook.com/cuetcomputerclub,CUET Computer Club
9,https://www.facebook.com/ieeecuetsb/,https://www.facebook.com/ieeecuetsb/posts/pfbi...,𝗘𝘅𝗽𝗹𝗼𝗿𝗶𝗻𝗴 𝗥𝗙 & 𝗛𝗶𝗴𝗵-𝗙𝗿𝗲𝗾𝘂𝗲𝗻𝗰𝘆 𝗘𝗻𝗴𝗶𝗻𝗲𝗲𝗿𝗶𝗻𝗴 | 𝗦𝗲...,Chittagong University,IEEE Student Branch CUET,False,False,False,[],[],,https://www.facebook.com/ieeecuetsb/,IEEE Student Branch CUET
